# Minimal Common Region (MCR) Analysis

Identify the **minimal genomic region on chromosome 19 that is consistently amplified across short-PFI patients** but not (or less so) in long-PFI patients.

InferCNV assigns HMM copy-number states to *regions* per subcluster.
Because subclusters represent distinct cell populations within a tumour (not every cell carries every CNV), we summarise at the **sample level**: for each 100 kb bin across chr19 we compute the **fraction of a sample's subclusters that carry a gain** at that position.

Author: Franziska Niemeyer

In [ ]:
DATA_DIR      = "../cnv_inference/infercnv_runs/combined_stroma_ref"

REGIONS_FILE   = DATA_DIR + "/HMM_CNV_predictions.HMMi6.leiden.hmm_mode-subclusters.Pnorm_0.5.pred_cnv_regions.dat"
GENES_FILE     = DATA_DIR + "/HMM_CNV_predictions.HMMi6.leiden.hmm_mode-subclusters.Pnorm_0.5.pred_cnv_genes.dat"
GROUPINGS_FILE = DATA_DIR + "/infercnv.17_HMM_predHMMi6.leiden.hmm_mode-subclusters.observation_groupings.txt"
ADATA_PATH     = "../../../quality_control/primary-cohort/adata.h5ad"

PFI_CAT_COL = "PFI"
SAMPLE_COL  = "patient"

PFI_ORDER   = ["short", "medium", "long"]
PFI_PALETTE = {'short': '#C7844A', 'medium': '#456EAE', 'long': '#538984'}

GENE_ID_COL     = "gene_ids"   # adata.var column with Ensembl IDs, or None
GENE_SYMBOL_COL = None         # adata.var column with symbols, or None

CHR            = "chr19"
CHR_LENGTH     = 58_617_616   # GRCh38
BIN_SIZE       = 100_000      # 100 kb bins  (reduce to 50_000 for higher resolution)
MIN_GAIN_STATE = 4            # 4=gain, 5=amp, 6=high-amp

SAMPLE_THRESHOLD = 0.1   # fraction of subclusters per sample that must show gain
N_SHORT_REQUIRED = 3     # minimum short-PFI samples meeting the threshold
DIFF_THRESHOLD   = 0.2   # minimum mean(short) - mean(long) gain fraction

HIGHLIGHT_GENES = {
    "ENSG00000130303": "BST2", 
    "ENSG00000105173": "CCNE1",
    "ENSG00000074181": "NOTCH3",
    "ENSG00000105221": "AKT2",
    "ENSG00000118046": "STK11",
    "ENSG00000085872": "CHERP",
    "ENSG00000099308": "MATK",
    "ENSG00000105058": "RAD23A",
}

OUT_DIR = "../outputs/mcr_results"
import os; os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy  as np
import pandas as pd
import matplotlib.pyplot  as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy.ndimage import uniform_filter1d   # for smoothing the profile
import anndata as ad

### Build Ensembl ID → gene symbol mapping

We extract the complete mapping directly from the AnnData object so every gene in the inferCNV output is annotated automatically — no manual list needed.

The `GENE_ID_COL` / `GENE_SYMBOL_COL` settings in the config cell define which `adata.var` columns hold the IDs and symbols respectively.
Setting either to `None` tells the notebook to use `adata.var.index` for that field.


In [ ]:
def build_gene_map(adata_path, gene_id_col, gene_symbol_col):
    """
    Build {ensembl_id: gene_symbol} from adata.var.

    Resolution order:
      IDs   : gene_id_col column if set, otherwise var.index
      Symbols: gene_symbol_col column if set, otherwise var.index
    """
    adata = ad.read_h5ad(adata_path)
    var   = adata.var.copy()

    id_series  = (var[gene_id_col].astype(str)
                  if gene_id_col and gene_id_col in var.columns
                  else pd.Series(var.index.astype(str), index=var.index))

    sym_series = (var[gene_symbol_col].astype(str)
                  if gene_symbol_col and gene_symbol_col in var.columns
                  else pd.Series(var.index.astype(str), index=var.index))

    mapping = dict(zip(id_series.values, sym_series.values))

    n_ensg = sum(1 for k in mapping if str(k).startswith("ENSG"))
    print(f"Gene map built from {adata_path}:")
    print(f"  {len(mapping):,} entries  |  {n_ensg:,} with ENSG IDs")
    if n_ensg == 0:
        print("  WARNING: no ENSG IDs found. Check GENE_ID_COL setting.")
        print(f"  adata.var columns : {list(adata.var.columns)}")
        print(f"  adata.var.index[:5]: {list(adata.var.index[:5])}")
    else:
        sample_ensg = [(k, v) for k, v in mapping.items() if str(k).startswith("ENSG")][:5]
        print("  Sample mappings:")
        for ensg, sym in sample_ensg:
            print(f"    {ensg}  ->  {sym}")
    return mapping

KNOWN_GENES = build_gene_map(ADATA_PATH, GENE_ID_COL, GENE_SYMBOL_COL)
print(f"\nKNOWN_GENES: {len(KNOWN_GENES):,} entries from adata.var")


### Load inferCNV files

In [ ]:
regions = pd.read_csv(REGIONS_FILE, sep="\t")
regions["subcluster"] = regions["cell_group_name"].str.split(".", n=1).str[-1]
chr_regions = regions[regions["chr"] == CHR].copy()

genes_df = pd.read_csv(GENES_FILE, sep="\t")
genes_df["subcluster"] = genes_df["cell_group_name"].str.split(".", n=1).str[-1]
chr_genes = genes_df[genes_df["chr"] == CHR].copy()

groupings = pd.read_csv(GROUPINGS_FILE, sep=" ", quotechar='"', index_col=0)
groupings.index.name = "barcode"
groupings = groupings.reset_index().rename(
    columns={"Dendrogram Group": "subcluster",
             "Annotation Group": "annotation_group"})
groupings["patient"] = groupings["subcluster"].str.extract(r"^(H\d+)")
sample_subs = groupings.groupby("patient")["subcluster"].unique()

print(f"{CHR} region rows   : {len(chr_regions):,}")
print(f"{CHR} gene rows     : {len(chr_genes):,}")
print(f"Total spots         : {len(groupings):,}")
print(f"Samples             : {sorted(sample_subs.index.tolist())}")
print()
print(f"{CHR} HMM state distribution:")
print(chr_regions["state"].value_counts().sort_index())


### Build per-bin gain-fraction matrix

For each 100 kb bin across chr19, and each subcluster, we record the maximum HMM state assigned to that position. We then aggregate to the sample level:
**gain fraction = fraction of a sample's subclusters with state ≥ MIN_GAIN_STATE at that bin**.

Subclusters with no chr19 entry in the regions file are assumed neutral (state 3).


In [ ]:
bins    = np.arange(0, CHR_LENGTH + BIN_SIZE, BIN_SIZE)
n_bins  = len(bins) - 1
bin_mids = (bins[:-1] + bins[1:]) / 2   # bin centre positions in bp

all_subs = chr_regions["subcluster"].unique()
sub_idx  = {s: i for i, s in enumerate(all_subs)}

# State matrix: n_bins x n_subclusters; default = 3 (neutral)
state_mat = np.full((n_bins, len(all_subs)), 3, dtype=np.float32)

for _, row in chr_regions.iterrows():
    si = sub_idx[row["subcluster"]]
    b0 = int(row["start"] // BIN_SIZE)
    b1 = min(int(row["end"]   // BIN_SIZE), n_bins - 1)
    state_mat[b0:b1+1, si] = np.maximum(state_mat[b0:b1+1, si], row["state"])

# Per-sample gain-fraction vector
samples = sorted(sample_subs.index)
gain_frac = {}     # sample -> np.array(n_bins)
max_state_vec = {} # sample -> np.array(n_bins)  [mean max state]

for s in samples:
    subs  = sample_subs[s]
    valid = [sub for sub in subs if sub in sub_idx]
    if not valid:
        gain_frac[s]     = np.zeros(n_bins)
        max_state_vec[s] = np.full(n_bins, 3.0)
        continue
    cols = [sub_idx[sub] for sub in valid]
    sm   = state_mat[:, cols]
    gain_frac[s]     = (sm >= MIN_GAIN_STATE).mean(axis=1)
    max_state_vec[s] = sm.max(axis=1).astype(float)

# Summary
print(f"Bin matrix: {n_bins} bins × {len(all_subs)} subclusters")
print(f"Bin size: {BIN_SIZE//1000} kb")
print()
print("Per-sample gain-fraction summary:")
print(f"  {'Sample':<8}  {'Max':>6}  {'Mean':>6}  {'Non-zero bins':>14}")
for s in samples:
    gf = gain_frac[s]
    print(f"  {s:<8}  {gf.max():.3f}  {gf.mean():.3f}  "
          f"{(gf>0).sum():>6} / {n_bins}")


### Load PFI categories

In [ ]:
adata = ad.read_h5ad(ADATA_PATH)
obs   = adata.obs[[PFI_CAT_COL]].copy()
if SAMPLE_COL and SAMPLE_COL in adata.obs.columns:
    obs["sample"] = adata.obs[SAMPLE_COL].astype(str)
else:
    obs["sample"] = adata.obs.index.str.extract(r"-(\d+)$")[0]
pfi_map = obs.groupby("sample")[PFI_CAT_COL].first().str.lower().str.strip().to_dict()

short_samples  = [s for s in samples if pfi_map.get(s) == "short"]
medium_samples = [s for s in samples if pfi_map.get(s) == "medium"]
long_samples   = [s for s in samples if pfi_map.get(s) == "long"]

print(f"Short  PFI ({len(short_samples)}): {short_samples}")
print(f"Medium PFI ({len(medium_samples)}): {medium_samples}")
print(f"Long   PFI ({len(long_samples)}): {long_samples}")

# Group mean gain-fraction vectors
short_mat  = np.array([gain_frac[s] for s in short_samples])   # n_short  x n_bins
medium_mat = np.array([gain_frac[s] for s in medium_samples])  # n_medium x n_bins
long_mat   = np.array([gain_frac[s] for s in long_samples])    # n_long   x n_bins

mean_short  = short_mat.mean(axis=0)
mean_long   = long_mat.mean(axis=0)
diff_score  = mean_short - mean_long  # positive = short-enriched

### Chr19 gain profile

Each row is a sample, coloured by PFI category. The x-axis is chr19 position.
The y-axis is the fraction of a sample's subclusters with a gain at that bin
(smoothed over a 3-bin window for clarity).
The bottom panel shows the differential score (short − long mean).


In [ ]:
SMOOTH = 3

from matplotlib.colors import to_rgb
import numpy as np

def make_shades(hex_color, n, lightness_range=(0.45, 1.0)):
    base  = np.array(to_rgb(hex_color))
    white = np.ones(3)
    alphas = np.linspace(lightness_range[1], lightness_range[0], n)
    return [tuple(a * base + (1 - a) * white) for a in alphas]

groups = {cat: [s for s in samples if pfi_map.get(s) == cat]
          for cat in PFI_ORDER}

sample_color = {}
for cat, grp in groups.items():
    shades = make_shades(PFI_PALETTE[cat], max(len(grp), 1))
    for s, c in zip(grp, shades):
        sample_color[s] = c

x_mb = bin_mids / 1e6

figs_profiles = []
axes_all = []

for cat in PFI_ORDER:
    grp = groups[cat]
    if not grp:
        continue

    fig_g, ax_g = plt.subplots(figsize=(10, 3))

    for s in grp:
        gf = uniform_filter1d(gain_frac[s], size=SMOOTH)
        ax_g.plot(x_mb, gf, color=sample_color[s],
                  linewidth=1.6, alpha=0.9, label=s)

    ax_g.set_xlim(0, CHR_LENGTH / 1e6)
    ax_g.set_ylim(0, 1.05)
    ax_g.set_yticks([0, 0.25, 0.5, 0.75, 1])
    ax_g.set_ylabel("Fraction of subclusters\nwith chr19 gain", fontsize=9)
    ax_g.tick_params(bottom=False, labelbottom=False)
    ax_g.spines["top"].set_visible(False)
    ax_g.spines["right"].set_visible(False)
    ax_g.legend(fontsize=8, loc="upper right", framealpha=0.85)
    ax_g.set_title(f"{CHR}  ·  PFI {cat}",
                   fontsize=12, fontweight="bold",
                   color=PFI_PALETTE[cat])

    figs_profiles.append(fig_g)
    axes_all.append(ax_g)
    plt.show()

fig_diff, ax_diff = plt.subplots(figsize=(10, 3))

diff_smooth = uniform_filter1d(diff_score, size=SMOOTH)
pos_mask    = diff_smooth >= 0
neg_mask    = diff_smooth < 0

ax_diff.fill_between(x_mb, diff_smooth, where=pos_mask,
                     color=PFI_PALETTE["short"], alpha=0.8,
                     linewidth=0, label="Short > long")
ax_diff.fill_between(x_mb, diff_smooth, where=neg_mask,
                     color=PFI_PALETTE["long"],  alpha=0.8,
                     linewidth=0, label="Long > short")
ax_diff.axhline(0, color="black", linewidth=0.6, alpha=0.5)
ax_diff.axhline( DIFF_THRESHOLD, color="black", linewidth=0.8,
                 linestyle="--", alpha=0.4,
                 label=f"Diff threshold ({DIFF_THRESHOLD})")
ax_diff.axhline(-DIFF_THRESHOLD, color="black", linewidth=0.8,
                 linestyle="--", alpha=0.4)
ax_diff.set_xlim(0, CHR_LENGTH / 1e6)
ax_diff.set_xlabel(f"{CHR} position (Mb)", fontsize=10)
ax_diff.set_ylabel("Short - long\n(mean gain fraction)", fontsize=8)
ax_diff.legend(fontsize=8, loc="upper right")
ax_diff.set_title(f"Differential copy-number gains on {CHR} - short vs long PFI", fontweight="bold")
ax_diff.spines["top"].set_visible(False)
ax_diff.spines["right"].set_visible(False)

for ensg, symbol in HIGHLIGHT_GENES.items():
    gene_rows = chr_genes[chr_genes["gene"] == ensg]
    if len(gene_rows):
        pos_mb = gene_rows["start"].iloc[0] / 1e6
        for ax in axes_all + [ax_diff]:
            ax.axvline(pos_mb, color="gray", linewidth=0.5,
                       linestyle=":", alpha=0.5, zorder=0)
        ax_diff.text(pos_mb, ax_diff.get_ylim()[1] * 0.85, symbol,
                     fontsize=6, rotation=90, ha="right", va="top",
                     color="gray")

plt.show()

for cat, fig_g in zip([c for c in PFI_ORDER if groups[c]], figs_profiles):
    fig_g.savefig(f"{OUT_DIR}/mcr_profile_{cat}.pdf", bbox_inches="tight")
    print(f"Saved: {OUT_DIR}/mcr_profile_{cat}.pdf")

fig_diff.savefig(f"{OUT_DIR}/mcr_profile_diff.pdf", bbox_inches="tight")
print(f"Saved: {OUT_DIR}/mcr_profile_diff.pdf")

### Call the minimal common region (MCR)

A bin is included in the MCR candidate set if:
1. **At least `N_SHORT_REQUIRED` short-PFI samples** have a gain fraction
   ≥ `SAMPLE_THRESHOLD` at that bin, AND
2. The **differential score** (mean short − mean long) exceeds `DIFF_THRESHOLD`.

Contiguous candidate bins are merged into MCR blocks. For each block we report:
- Genomic coordinates
- Number and fraction of short-PFI samples covered
- Mean gain fraction in short vs long PFI
- InferCNV genes falling within the block


In [ ]:
# ── Bin-level calls ────────────────────────────────────────────────────────
n_short_with_gain = (short_mat >= SAMPLE_THRESHOLD).sum(axis=0)  # n_bins
mcr_mask = (
    (n_short_with_gain >= N_SHORT_REQUIRED) &
    (diff_score >= DIFF_THRESHOLD)
)

print(f"MCR parameters:")
print(f"  Sample threshold  : gain_frac >= {SAMPLE_THRESHOLD} in >= {N_SHORT_REQUIRED}/{len(short_samples)} short-PFI samples")
print(f"  Diff threshold    : mean(short) - mean(long) >= {DIFF_THRESHOLD}")
print(f"  Candidate bins    : {mcr_mask.sum()} "
      f"({mcr_mask.sum() * BIN_SIZE / 1e6:.2f} Mb)")
print()

# ── Merge contiguous bins into blocks ─────────────────────────────────────
def merge_bins(mask, bin_size):
    blocks = []
    in_block = False
    for i, m in enumerate(mask):
        if m and not in_block:
            start = i * bin_size
            in_block = True
        elif not m and in_block:
            blocks.append((start, i * bin_size))
            in_block = False
    if in_block:
        blocks.append((start, len(mask) * bin_size))
    return blocks

mcr_blocks = merge_bins(mcr_mask, BIN_SIZE)

print(f"MCR blocks ({len(mcr_blocks)} contiguous regions):")
print()

mcr_records = []
for b_start, b_end in mcr_blocks:
    b0 = b_start // BIN_SIZE
    b1 = min(b_end // BIN_SIZE, n_bins - 1)

    # Per-sample gain fractions within this block
    short_block = np.array([gain_frac[s][b0:b1+1].mean() for s in short_samples])
    long_block  = np.array([gain_frac[s][b0:b1+1].mean() for s in long_samples])
    n_short_pass = (short_block >= SAMPLE_THRESHOLD).sum()

    # Genes within block
    block_genes = chr_genes[
        (chr_genes["start"] <= b_end) &
        (chr_genes["end"]   >= b_start)
    ]["gene"].unique()
    known = [KNOWN_GENES[g] for g in block_genes if g in KNOWN_GENES]
    n_genes = len(block_genes)

    mcr_records.append({
        "chr"          : CHR,
        "start"        : b_start,
        "end"          : b_end,
        "size_kb"      : (b_end - b_start) // 1000,
        "n_short_pass" : int(n_short_pass),
        "n_short_total": len(short_samples),
        "mean_gain_short": round(float(short_block.mean()), 4),
        "mean_gain_long" : round(float(long_block.mean()) if len(long_block) else 0, 4),
        "diff"           : round(float(short_block.mean() - long_block.mean()) if len(long_block) else float(short_block.mean()), 4),
        "n_inferCNV_genes": n_genes,
        "known_genes"    : "; ".join(known) if known else "—",
    })

    print(f"  {CHR}:{b_start:>12,} – {b_end:>12,}  ({(b_end-b_start)//1000} kb)")
    print(f"    Short-PFI samples covered : {n_short_pass}/{len(short_samples)}")
    for s, gf in zip(short_samples, short_block):
        print(f"      {s}: gain_frac={gf:.3f}")
    long_str = f"{long_block.mean():.3f}" if len(long_block) else "N/A"
    print(f"    Mean gain short / long    : {short_block.mean():.3f} / {long_str}")
    print(f"    InferCNV genes in block   : {n_genes}  "
          f"({'known: ' + ', '.join(known) if known else 'none in KNOWN_GENES'})")
    print()

mcr_df = pd.DataFrame(mcr_records)
if len(mcr_df):
    mcr_df.to_csv(f"{OUT_DIR}/mcr_blocks.tsv", sep="\t", index=False)
    print(f"Saved: {OUT_DIR}/mcr_blocks.tsv")

    # BED file for genome browsers (UCSC / IGV)
    bed_df = mcr_df[["chr","start","end"]].copy()
    bed_df["name"]  = [f"MCR_{i+1}" for i in range(len(bed_df))]
    bed_df["score"] = (mcr_df["diff"] * 1000).clip(0, 1000).astype(int)
    bed_df.to_csv(f"{OUT_DIR}/mcr_blocks.bed", sep="\t", index=False, header=False)
    print(f"Saved: {OUT_DIR}/mcr_blocks.bed  (load in IGV or UCSC)")


### Sensitivity analysis — MCR size across thresholds

Because the MCR boundaries depend on the choice of `SAMPLE_THRESHOLD` and
`N_SHORT_REQUIRED`, we sweep a grid of values to show how the MCR size changes.
This helps choose a biologically reasonable threshold.


In [ ]:
thresholds  = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]
n_required  = list(range(2, len(short_samples) + 1))

# Build grid: MCR size (Mb) for each (threshold, n_required) combination
size_grid = np.zeros((len(thresholds), len(n_required)))

for ti, t in enumerate(thresholds):
    for ni, nr in enumerate(n_required):
        mask = (
            ((short_mat >= t).sum(axis=0) >= nr) &
            (diff_score >= DIFF_THRESHOLD)
        )
        size_grid[ti, ni] = mask.sum() * BIN_SIZE / 1e6

# Plot heatmap
fig, ax = plt.subplots(figsize=(max(5, len(n_required) * 1.2), 4))
im = ax.imshow(size_grid, aspect="auto", cmap="YlOrRd",
               vmin=0, vmax=size_grid.max())
ax.set_xticks(range(len(n_required)))
ax.set_xticklabels([f">={n}" for n in n_required], fontsize=9)
ax.set_yticks(range(len(thresholds)))
ax.set_yticklabels([str(t) for t in thresholds], fontsize=9)
ax.set_xlabel(f"Short-PFI samples required (of {len(short_samples)})", fontsize=10)
ax.set_ylabel("Gain-fraction threshold\nper sample", fontsize=10)
ax.set_title("MCR size (Mb) across threshold combinations",
             fontsize=11, fontweight="bold")

# Annotate cells
for ti in range(len(thresholds)):
    for ni in range(len(n_required)):
        val = size_grid[ti, ni]
        ax.text(ni, ti, f"{val:.1f}", ha="center", va="center",
                fontsize=8, color="black" if val < size_grid.max()*0.7 else "white")

# Mark current parameters
try:
    ti_cur = thresholds.index(SAMPLE_THRESHOLD)
    ni_cur = n_required.index(N_SHORT_REQUIRED)
    ax.add_patch(mpatches.Rectangle(
        (ni_cur - 0.5, ti_cur - 0.5), 1, 1,
        fill=False, edgecolor="blue", linewidth=2.5, label="current params"
    ))
except ValueError:
    pass

plt.colorbar(im, ax=ax, label="Minimal common region (MCR) size (Mb)", shrink=0.8)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/mcr_02_sensitivity_grid.pdf", bbox_inches="tight")
plt.show()
print(f"Saved: {OUT_DIR}/mcr_02_sensitivity_grid.pdf")

### Gain-fraction heatmap — bins × samples

Rows are 100 kb bins (only the informative subset: bins with mean gain > 0.05 in at least one sample). Columns are samples, clustered by profile similarity. The MCR bins are highlighted with a red bar on the left.

In [ ]:
# Filter to informative bins
all_gain_mat = np.array([gain_frac[s] for s in samples])   # n_samples x n_bins
informative  = all_gain_mat.max(axis=0) >= 0.05
bin_labels   = [f"{b*BIN_SIZE//1_000_000:.1f}" for b in range(n_bins)]

hm_df = pd.DataFrame(
    all_gain_mat[:, informative].T,
    index  = [f"{b*BIN_SIZE/1e6:.2f}" for b in np.where(informative)[0]],
    columns= samples,
)

# Row colours: is this bin in the MCR?
row_colors = pd.Series(
    ["#C0392B" if mcr_mask[b] else "#EEEEEE"
     for b in np.where(informative)[0]],
    index=hm_df.index,
    name="MCR"
)

# Column colours: PFI category
col_colors = pd.Series(
    [PFI_PALETTE[pfi_map.get(s, "medium")] for s in samples],
    index=samples,
    name="PFI"
)

g = sns.clustermap(
    hm_df,
    row_cluster=False,    # keep genomic order on y
    col_cluster=True,
    cmap="RdBu_r", vmin=0, vmax=1,
    row_colors=row_colors,
    col_colors=col_colors,
    figsize=(3, 7),
    linewidths=0,
    cbar_kws={"label": "Gain fraction", "shrink": 0.4},
    xticklabels=True,
    yticklabels=False,
)
g.ax_heatmap.set_xlabel("Sample", fontsize=10)
g.ax_heatmap.set_ylabel(f"{CHR} position (100 kb bins)", fontsize=10)
g.ax_heatmap.set_title(f"{CHR} gain-fractions",
               fontsize=12, fontweight="bold", y=1.35)

# Add legend
legend_patches = [
    mpatches.Patch(facecolor=PFI_PALETTE[c], label=f"PFI: {c}") for c in PFI_ORDER
] + [
    mpatches.Patch(facecolor="#C0392B", label="MCR bin"),
    mpatches.Patch(facecolor="#EEEEEE", label="Non-MCR bin"),
]
g.fig.legend(handles=legend_patches, loc="center left",
             bbox_to_anchor=(1, 0.5), fontsize=8, title="Legend")

plt.savefig(f"{OUT_DIR}/mcr_03_heatmap.pdf", bbox_inches="tight")
plt.show()
print(f"Saved: {OUT_DIR}/mcr_03_heatmap.pdf")


### Annotate MCR with gene content

Report all inferCNV genes that fall within MCR blocks, along with their mean HMM state in short-PFI vs long-PFI subclusters.
Known cancer genes (from `HIGHLIGHT_GENES`) are flagged.

In [ ]:
if len(mcr_blocks) == 0:
    print("No MCR blocks called — try relaxing SAMPLE_THRESHOLD or N_SHORT_REQUIRED.")
else:
    gene_records = []
    for b_start, b_end in mcr_blocks:
        block_gene_rows = chr_genes[
            (chr_genes["start"] <= b_end) &
            (chr_genes["end"]   >= b_start)
        ].copy()
        block_gene_rows["symbol"] = block_gene_rows["gene"].map(
            lambda g: HIGHLIGHT_GENES.get(g, "")
        )
        block_gene_rows["mcr_start"] = b_start
        block_gene_rows["mcr_end"]   = b_end
        gene_records.append(block_gene_rows)

    if gene_records:
        all_mcr_genes = pd.concat(gene_records, ignore_index=True)

        # Mean state per gene in short vs long subclusters
        short_subs_list = [s for sub in short_samples for s in sample_subs[sub]
                           if s in sub_idx]
        long_subs_list  = [s for sub in long_samples  for s in sample_subs[sub]
                           if s in sub_idx]

        def mean_state_group(ensg, sub_list):
            rows = chr_genes[
                (chr_genes["gene"] == ensg) &
                (chr_genes["subcluster"].isin(sub_list))
            ]
            return rows["state"].mean() if len(rows) else np.nan

        unique_genes = all_mcr_genes.drop_duplicates("gene")[["gene","symbol","start","end","mcr_start","mcr_end"]]
        unique_genes = unique_genes.sort_values("start").copy()
        unique_genes["mean_state_short"] = unique_genes["gene"].apply(
            lambda g: mean_state_group(g, short_subs_list))
        unique_genes["mean_state_long"]  = unique_genes["gene"].apply(
            lambda g: mean_state_group(g, long_subs_list))
        unique_genes["state_diff"] = (
            unique_genes["mean_state_short"] - unique_genes["mean_state_long"]
        )
        unique_genes["is_known"] = unique_genes["gene"].isin(HIGHLIGHT_GENES)

        print(f"Genes within MCR blocks: {len(unique_genes)}")
        print(f"Known cancer genes: {unique_genes['is_known'].sum()}")
        print()
        print("Top 20 by state_diff (short - long mean HMM state):")
        display_cols = ["symbol","gene","start","end","mean_state_short",
                        "mean_state_long","state_diff","is_known"]
        print(unique_genes.sort_values("state_diff", ascending=False)
              .head(20)[display_cols].to_string(index=False))

        unique_genes.to_csv(f"{OUT_DIR}/mcr_genes.tsv", sep="\t", index=False)
        print(f"\nSaved: {OUT_DIR}/mcr_genes.tsv")
    else:
        print("No genes found within MCR blocks.")


### Zoom-in on the MCR — gene-level annotation plot

Plots the chr19 profile zoomed to ± 2 Mb around the largest MCR block, with individual gene positions shown as ticks at the bottom.

In [ ]:
if len(mcr_blocks) == 0:
    print("No MCR blocks — skipping zoom plot.")
else:
    # Zoom to largest MCR block
    largest = max(mcr_blocks, key=lambda b: b[1] - b[0])
    zoom_start = max(0,          largest[0] - 2_000_000)
    zoom_end   = min(CHR_LENGTH, largest[1] + 2_000_000)

    zoom_bins  = (bin_mids >= zoom_start) & (bin_mids <= zoom_end)
    x_zoom     = bin_mids[zoom_bins] / 1e6

    fig, axes = plt.subplots(
        len(samples) + 1, 1,
        figsize=(14, 2.5 * len(samples) + 2.5),
        sharex=True,
        gridspec_kw={"height_ratios": [1]*len(samples) + [0.8], "hspace": 0.05}
    )

    for i, s in enumerate(samples):
        ax = axes[i]
        gf = uniform_filter1d(gain_frac[s], size=SMOOTH)[zoom_bins]
        color = PFI_PALETTE[pfi_map.get(s, "medium")]
        ax.fill_between(x_zoom, gf, alpha=0.7, color=color, linewidth=0)
        ax.plot(x_zoom, gf, color=color, linewidth=0.8)

        # Shade MCR
        for b_start, b_end in mcr_blocks:
            if b_start < zoom_end and b_end > zoom_start:
                ax.axvspan(b_start/1e6, b_end/1e6, color="#C0392B", alpha=0.1, zorder=0)

        ax.set_ylim(0, 1.05)
        ax.set_yticks([0, 0.5, 1])
        ax.set_yticklabels(["0","0.5","1"], fontsize=7)
        ax.set_ylabel(s, fontsize=8, rotation=0, ha="right", va="center", labelpad=60)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    # Gene track at bottom
    ax_genes = axes[-1]
    ax_genes.set_ylim(0, 1)
    ax_genes.set_yticks([])
    ax_genes.set_ylabel("Genes", fontsize=8, rotation=0,
                         ha="right", va="center", labelpad=60)
    ax_genes.spines["top"].set_visible(False)
    ax_genes.spines["right"].set_visible(False)

    # Plot all inferCNV genes in zoom region
    zoom_gene_rows = chr_genes[
        (chr_genes["start"] >= zoom_start) &
        (chr_genes["end"]   <= zoom_end)
    ].drop_duplicates("gene")

    for _, grow in zoom_gene_rows.iterrows():
        mid_mb = (grow["start"] + grow["end"]) / 2e6
        ax_genes.plot([grow["start"]/1e6, grow["end"]/1e6], [0.5, 0.5],
                      color="steelblue", linewidth=3, solid_capstyle="butt", alpha=0.7)
        symbol = KNOWN_GENES.get(grow["gene"], "")
        if symbol:
            ax_genes.text(mid_mb, 0.15, symbol, fontsize=7, ha="center",
                          va="top", rotation=45, color="#C0392B", fontweight="bold")

    # Shade MCR in gene track
    for b_start, b_end in mcr_blocks:
        if b_start < zoom_end and b_end > zoom_start:
            ax_genes.axvspan(b_start/1e6, b_end/1e6, color="#C0392B", alpha=0.15, zorder=0)

    ax_genes.set_xlabel(f"{CHR} position (Mb)", fontsize=10)
    axes[0].set_title(
        f"{CHR} MCR zoom  ({zoom_start//1_000_000:.1f}–{zoom_end//1_000_000:.1f} Mb)\n"
        f"Red shading = MCR ({largest[0]//1_000_000:.1f}–{largest[1]//1_000_000:.1f} Mb)",
        fontsize=12, fontweight="bold"
    )

    plt.savefig(f"{OUT_DIR}/mcr_04_zoom_plot.pdf", bbox_inches="tight")
    plt.show()

### Summary

In [ ]:
import glob as _glob

print("=" * 68)
print("  MINIMAL COMMON REGION SUMMARY")
print("=" * 68)
print(f"  Chromosome     : {CHR}  ({CHR_LENGTH/1e6:.1f} Mb total)")
print(f"  Bin size       : {BIN_SIZE//1000} kb  ({n_bins} bins)")
print(f"  Short PFI ({len(short_samples)})  : {short_samples}")
print(f"  Long  PFI ({len(long_samples)})  : {long_samples}")
print()
print(f"  MCR parameters :")
print(f"    Gain threshold       : {SAMPLE_THRESHOLD} (fraction of subclusters per sample)")
print(f"    Short samples req.   : {N_SHORT_REQUIRED}/{len(short_samples)}")
print(f"    Differential filter  : short - long >= {DIFF_THRESHOLD}")
print()

if len(mcr_blocks):
    total_mcr_bp = sum(e-s for s, e in mcr_blocks)
    print(f"  MCR blocks found : {len(mcr_blocks)}")
    print(f"  Total MCR size   : {total_mcr_bp/1e6:.2f} Mb")
    print()
    for i, (bs, be) in enumerate(mcr_blocks):
        block_genes = chr_genes[
            (chr_genes["start"] <= be) & (chr_genes["end"] >= bs)
        ]["gene"].unique()
        known = [KNOWN_GENES[g] for g in block_genes if g in KNOWN_GENES]
        print(f"  Block {i+1}: {CHR}:{bs:,}–{be:,}  "
              f"({(be-bs)//1000} kb)")
        if known:
            print(f"    Known genes: {', '.join(known)}")
else:
    print("  No MCR blocks called with current parameters.")
    print("  Suggestions:")
    print("    → Lower SAMPLE_THRESHOLD (try 0.05)")
    print("    → Lower N_SHORT_REQUIRED (try 2)")
    print("    → Lower DIFF_THRESHOLD (try 0.10)")